> 在线讲义：[第 11 章 · MoE：专家路由与批内权重复用](https://wikieden.github.io/ai-infra-from-constraints/lessons/11-moe.html) · 本书《从约束推导 AI Infra》

# 第 11 章 · MoE：专家路由与批内权重复用

配套讲义：仓库里 `ai-infra/notes/lessons/11-moe.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Qwen3.6 的专家 | $3df$ | `expert_params(2048, 512)` | 3,145,728，6 MiB；每层 1.5 GiB，8 个 48 MiB |
| 第二步 两种分派 | $t_e$ 列表 | `route_uniform(64, 8, 256)`、`route_concentrated(64, 8)` | $U$ = 256 / 8，每专家 2 / 64 个 |
| 第二步 运算与读取 | $2N_e\sum t_e$、$b N_e U$ | `expert_cost(te, N_e)` | 运算相同；读取 1.5 GiB vs 48 MiB |
| 第三步 V4-Flash 专家 | $3 \times 4096 \times 2048$ | `expert_params(4096, 2048)` | 25,165,824 |

Qwen3.6-35B-A3B（d = 2048、256 个专家、选 8 个、专家宽 512、40 层）与 V4-Flash（d = 4096、专家宽 2048）取自 AII 第 2 章 §2.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `expert_params`**：一个 SwiGLU 专家的参数：gate、up、down 共 3 × d × f。

In [ ]:
def expert_params(d: int, f: int) -> int:
    """一个 SwiGLU 专家的参数：gate、up、down 共 3 × d × f。"""
    return 3 * d * f

**函数 `route_uniform`**：把 B·k 次分派轮流摊到 E 个专家：返回每个专家收到的 token 数 t_e。

In [ ]:
def route_uniform(B: int, k: int, E: int) -> list[int]:
    """把 B·k 次分派轮流摊到 E 个专家：返回每个专家收到的 token 数 t_e。"""
    te = [0] * E
    for i in range(B * k):
        te[i % E] += 1
    return te

**函数 `route_concentrated`**：所有 token 都选同一组 k 个专家。

In [ ]:
def route_concentrated(B: int, k: int, E: int = 256) -> list[int]:
    """所有 token 都选同一组 k 个专家。"""
    return [B if e < k else 0 for e in range(E)]

**函数 `expert_cost`**：(专家运算 2·N_e·Σt_e, 权重读取 b·N_e·U)，U = 实际访问的不同专家数。

In [ ]:
def expert_cost(te: list[int], n_e: int, b: int = 2) -> tuple[int, int]:
    """(专家运算 2·N_e·Σt_e, 权重读取 b·N_e·U)，U = 实际访问的不同专家数。"""
    flops = 2 * n_e * sum(te)  # 按分派的总行数
    u = sum(1 for t in te if t > 0)  # 实际访问的不同专家数
    return flops, b * n_e * u  # 读取按不同专家数

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Qwen3.6 的路由专家")
n_e = expert_params(2048, 512)
print(f"  一个专家 {n_e:,} 个参数 = {n_e * 2 / 2**20:.0f} MiB（BF16）")
print(f"  每层 256 个 = {256 * n_e * 2 / 2**30:.1f} GiB；一个 token 选中的 8 个 = {8 * n_e * 2 / 2**20:.0f} MiB")

In [ ]:
print("\n第二步：B = 64，每 token 选 8 个，共 512 次分派")
for name, te in (("均匀", route_uniform(64, 8, 256)), ("集中", route_concentrated(64, 8))):
    flops, read = expert_cost(te, n_e)
    u = sum(1 for t in te if t > 0)
    print(f"  {name}：U = {u}，每专家 {max(te)} 个 token，运算 {flops / 1e9:.2f} GFLOPs，读 {read / 2**20:.0f} MiB")
print(f"  40 层：全部存 {40 * 256 * n_e * 2 / 2**30:.0f} GiB，集中时每批读 {40 * 8 * n_e * 2 / 2**30:.3f} GiB")

In [ ]:
print("\n第三步：V4-Flash 的一个路由专家")
print(f"  3 × 4096 × 2048 = {expert_params(4096, 2048):,}")

## 3. 代码里才看得到的

1. `route_uniform` 按轮流的方式把 512 次分派摊到 256 个专家，正好每个 2 个；真实路由不会这么均匀，这只是上限情形。
2. 两种分派的 `sum(te)` 都是 512：运算量只取决于分派总数，测试里专门核对。
3. 读取量假设每个被选中的专家整份读一次（理想复用）；真实 kernel 的分块方式可能让大专家被读不止一次（第 26 章）。

## 4. 自己改一改

- 把 `route_concentrated` 改成集中到 32 个专家，看 $U$ 和读取量。
- 把 batch 从 64 改成 8，均匀分派时还能覆盖全部 256 个专家吗？
- 写一个随机路由（每个 token 随机选 8 个不同专家），多跑几次看 $U$ 的平均值。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 医生养得多，病人挂得少；
> 算量看分派，读量看开了几扇门；
> 分诊台人人过，专科先摘要再还原。